# 2.4 Apache Airflow — Pipeline Orchestration> ☕ **Oozie parallel:** Airflow is the Python-native successor to Oozie. DAGs replace XML workflows.| Oozie | Airflow ||-------|----------|| XML workflow definition | Python DAG file || Coordinator | Schedule/Timetable || Actions (Hive, Spark) | Operators (BashOperator, SparkSubmitOperator) || `${variable}` | Jinja templates `{{ var }}` |> ⚠️ Airflow runs as a service — this notebook shows DAG **authoring**, not execution.---

In [ ]:
# Example DAG — save as dags/healthcare_etl.py in your Airflow homefrom datetime import datetime, timedeltafrom airflow import DAGfrom airflow.operators.python import PythonOperatorfrom airflow.operators.bash import BashOperatorfrom airflow.providers.apache.spark.operators.spark_submit import SparkSubmitOperatordefault_args = {    "owner": "vitalii",    "depends_on_past": False,    "retries": 2,    "retry_delay": timedelta(minutes=5),    "email_on_failure": True,}with DAG(    dag_id="healthcare_etl",    default_args=default_args,    description="Daily healthcare data pipeline",    schedule="0 6 * * *",  # daily at 6 AM    start_date=datetime(2024, 1, 1),    catchup=False,    tags=["healthcare", "etl"],) as dag:    extract = BashOperator(        task_id="extract_from_source",        bash_command="echo 'Extracting data from SFTP...'",    )    def validate_data(**context):        """Validate extracted data quality."""        print(f"Validating data for {context['ds']}")        # In practice: run Great Expectations checks        return {"valid_records": 50000, "invalid": 12}    validate = PythonOperator(        task_id="validate_data",        python_callable=validate_data,    )    transform = SparkSubmitOperator(        task_id="transform_with_spark",        application="/opt/spark/jobs/transform.py",        conn_id="spark_default",    )    load = BashOperator(        task_id="load_to_warehouse",        bash_command="echo 'Loading to data warehouse...'",    )    # Define dependencies — like Oozie's fork/join    extract >> validate >> transform >> loadprint("DAG defined successfully")print(f"Tasks: {[t.task_id for t in dag.tasks]}")

## Key Airflow Concepts for Oozie Users1. **TaskFlow API** — modern way to write DAGs with decorated Python functions2. **XComs** — pass data between tasks (like Oozie EL functions)3. **Sensors** — wait for external conditions (file exists, API available)4. **Dynamic DAGs** — generate tasks programmatically (no XML templating needed)

In [ ]:
# TaskFlow API — cleaner than PythonOperatorfrom airflow.decorators import dag, task@dag(    schedule="@daily",    start_date=datetime(2024, 1, 1),    catchup=False,)def healthcare_etl_v2():        @task    def extract() -> list[dict]:        """Extract patient records."""        return [{"id": "P001", "age": 45}, {"id": "P002", "age": 62}]        @task    def transform(records: list[dict]) -> list[dict]:        """Add derived fields."""        for r in records:            r["age_group"] = "senior" if r["age"] >= 65 else "adult"        return records        @task    def load(records: list[dict]):        """Load to destination."""        print(f"Loading {len(records)} records")        # Chain with Python — data passes via XCom automatically    raw = extract()    transformed = transform(raw)    load(transformed)# Instantiate the DAGhealthcare_dag = healthcare_etl_v2()print("TaskFlow DAG created")